# 05 - Evaluation and Attention Visualization
Evaluate on unseen prompts and inspect cross-attention behavior.

In [ ]:
from setup_01 import *
from data_02 import data_load
from model_03 import Qwen35Vision
import os
import json
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

EVAL_CFG = {
    "model_size": CONFIG.get("train_session", {}).get("model_size", "0.8b"),
    "eval_subset_size": 3,
    "test_size": 0.34,
    "gen_max_new_tokens": 8,
    "output_dir": os.path.join("outputs", "eval_attention"),
}

class EvalAttentionManager:
    def __init__(self, eval_cfg=None):
        self.cfg = eval_cfg if eval_cfg is not None else EVAL_CFG
        self.model_size = self.cfg.get("model_size", "0.8b")
        self.output_dir = self.cfg.get("output_dir", os.path.join("outputs", "eval_attention"))
        self.train_log_dir = os.path.join("outputs", "train_logs", self.model_size)

        os.makedirs(self.output_dir, exist_ok=True)
        set_seed(CONFIG["seed"])

        self.model = None
        self.dataset_test = None
        self.dataset_gen_test = None

    def load_model(self, filename="latest"):
        self.model = Qwen35Vision(model_size=self.model_size)
        self.model.model_load(Install=False, filename=filename)
        return self.model

    def prepare_eval_data(self):
        dataset = data_load()
        subset_size = min(self.cfg.get("eval_subset_size", 3), len(dataset))
        subset = dataset.select(range(subset_size))

        _, self.dataset_test = self.model.data_divide(
            subset,
            test_size=self.cfg.get("test_size", 0.34),
            seed=CONFIG["seed"],
        )
        self.dataset_gen_test = self.model.data_divide_gen(
            subset,
            test_size=self.cfg.get("test_size", 0.34),
            seed=CONFIG["seed"],
        )

    def run_eval(self):
        eval_metrics = self.model.eval_run(self.dataset_test)
        gen_metrics = self.model.gen_eval_run(
            self.dataset_gen_test,
            max_new_tokens=self.cfg.get("gen_max_new_tokens", 8),
        )

        eval_result = {
            "eval_metrics": eval_metrics,
            "gen_metrics": gen_metrics,
            "config": self.cfg,
            "timestamp": datetime.now().isoformat(),
        }

        result_path = os.path.join(self.output_dir, "eval_metrics.json")
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(eval_result, f, indent=2)

        print("Eval summary:", eval_result)
        print("Saved metrics:", result_path)
        return eval_result

    def plot_training_from_logs(self, metrics_csv_path=None):
        if metrics_csv_path is None:
            pattern = os.path.join(self.train_log_dir, "metrics_by_epoch_*.csv")
            matches = sorted(glob.glob(pattern))
            if not matches:
                print(f"No metrics CSV found under: {self.train_log_dir}")
                return None
            metrics_csv_path = matches[-1]

        epochs = []
        train_loss = []
        val_loss = []

        with open(metrics_csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                epoch_raw = row.get("epoch", "").strip()
                train_raw = row.get("train_loss", "").strip()
                val_raw = row.get("val_loss", "").strip()

                if epoch_raw == "":
                    continue
                epochs.append(int(epoch_raw))
                train_loss.append(float(train_raw) if train_raw != "" else np.nan)
                val_loss.append(float(val_raw) if val_raw != "" else np.nan)

        if not epochs:
            print(f"No usable rows in: {metrics_csv_path}")
            return None

        fig_path = os.path.join(self.output_dir, "training_curve_from_logs.png")
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, train_loss, marker="o", label="train_loss")
        plt.plot(epochs, val_loss, marker="s", label="val_loss")
        plt.title("Training Curve From Logs")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_path, dpi=160)
        plt.close()

        print(f"Read metrics CSV: {metrics_csv_path}")
        print(f"Saved training curve: {fig_path}")
        return fig_path

    def save_attention_plot(self, sample_idx=0, max_tokens=60):
        if self.dataset_test is None:
            raise RuntimeError("Call prepare_eval_data() first.")

        n_samples = self.dataset_test["input_ids"].shape[0]
        if n_samples == 0:
            raise ValueError("dataset_test is empty. Increase eval_subset_size.")

        idx = min(sample_idx, n_samples - 1)
        input_ids = self.dataset_test["input_ids"][idx:idx+1].to(self.model.device)
        attention_mask = self.dataset_test["attention_mask"][idx:idx+1].to(self.model.device)
        pixel_values = self.dataset_test["pixel_values"][idx:idx+1].to(self.model.device)
        image_grid_thw = self.dataset_test["image_grid_thw"][idx:idx+1].to(self.model.device)
        labels = self.dataset_test["labels"][idx].clone()

        self.model.model.eval()
        with torch.no_grad():
            outputs = self.model.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                output_attentions=True,
                return_dict=True,
            )

        attentions = outputs.attentions
        if attentions is None or len(attentions) == 0:
            print("No attentions returned by model. Skip plotting.")
            return None

        last_layer_attn = attentions[-1][0].float().cpu()
        mean_attn = last_layer_attn.mean(dim=0)

        target_positions = torch.where(labels != -100)[0]
        query_pos = int(target_positions[0].item()) if len(target_positions) > 0 else int(mean_attn.shape[0] - 1)

        seq_len = int(mean_attn.shape[1])
        cut = min(max_tokens, seq_len)
        attn_vec = mean_attn[query_pos, :cut].numpy()

        token_ids = input_ids[0, :cut].detach().cpu().tolist()
        tokens = self.model.tokenizer.convert_ids_to_tokens(token_ids)
        token_labels = [t if len(t) < 18 else t[:15] + "..." for t in tokens]

        fig_path = os.path.join(self.output_dir, f"attention_sample_{idx}.png")
        plt.figure(figsize=(14, 5))
        plt.plot(np.arange(cut), attn_vec, linewidth=1.5)
        plt.title(f"Last-layer mean attention from query token index {query_pos}")
        plt.xlabel("Key token position")
        plt.ylabel("Attention weight")
        plt.grid(alpha=0.25)

        step = max(1, cut // 12)
        tick_pos = np.arange(0, cut, step)
        plt.xticks(tick_pos, [token_labels[p] for p in tick_pos], rotation=45, ha="right")
        plt.tight_layout()
        plt.savefig(fig_path, dpi=160)
        plt.close()

        meta = {
            "sample_idx": idx,
            "query_pos": query_pos,
            "sequence_len": seq_len,
            "plotted_tokens": cut,
            "figure": fig_path,
        }
        meta_path = os.path.join(self.output_dir, f"attention_sample_{idx}.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        print("Saved attention figure:", fig_path)
        print("Saved attention meta:", meta_path)
        return fig_path



In [ ]:
if __name__ == "__main__":
    # Optional: rerun only this cell when you want a fresh curve from latest training logs.
    mgr = EvalAttentionManager(EVAL_CFG)
    mgr.load_model(filename="latest")
    mgr.prepare_eval_data()
    eval_result = mgr.run_eval()
    training_curve = mgr.plot_training_from_logs()
    attention_figure = mgr.save_attention_plot(sample_idx=0, max_tokens=60)
    training_curve = mgr.plot_training_from_logs()
    print("Training curve from logs:", training_curve)

## Summary
This notebook evaluates the fine-tuned Qwen3.5-Vision model on a small unseen subset and visualizes one attention profile for interpretation.

### What Was Done
- Loaded the fine-tuned checkpoint with the same model size used during training.
- Built a small test split from the saved dataset.
- Ran two evaluation modes:
  - Standard forward evaluation (`eval_run`): validation loss, token accuracy, perplexity.
  - Generation evaluation (`gen_eval_run`): generation token accuracy and exact match.
- Extracted last-layer attention from one sample and saved a visualization.

### Output Files
- `outputs/eval_attention/eval_metrics.json`: combined eval and generation metrics.
- `outputs/eval_attention/attention_sample_0.png`: attention curve figure.
- `outputs/eval_attention/attention_sample_0.json`: metadata for the plotted sample.

### How To Read Results
- Lower `val_loss` and lower `perplexity` indicate better fit.
- Higher `token_acc` and `gen_token_acc` indicate better token-level quality.
- `gen_exact_match` is strict and may stay low even when outputs are mostly correct.
- Attention plots are diagnostic: use them to compare model focus patterns across runs.